In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks/AI/'

# 2. CNN 모델 구현하기

## 2-1. 데이터 준비

- 모델 학습을 위한 데이터 준비
- 이번 실습에서는 `CIFAR-10` 이미지 데이터셋을 사용할 예
    1. 32 x 32 크기의 이미지 6만장
    2. 색상채널 3개 (RGB)
    3. 클래스 수 10개
        - 비행기, 자동차, 새, 고양이, 사슴, 개, 개구리, 말, 배, 트럭
    4. 데이터 분할 구성
        - 훈련 세트 5만장 (클래스당 5천장)
        - 테스트 세트 1만장 (클래스당 1천장)

### 2-1-1. 데이터 로드 및 전처리

- torchvision 라이브러리로 데이터셋 불러오기 및 전처리 진행
1. `torchvision.datasets.CIFAR10`
    - CIFAR-10 데이터셋 로드
2. `transforms.ToTensor()`
    - 파이썬 이미지 라이브러리(PIL) 형식의 이미지를 텐서로 변환
    - 픽셀 값의 범위가 [0, 255]에서 [0.0, 1.0]으로 조정
3. `transforms.Normalize()`
    - 텐서의 픽셀 값을 평균과 표준편차로 정규화
    - **각 채널**의 데이터 분포를 평균 0, 표준편차 1에 가깝게 정규화
        - CIFAR-10과 같이 자주 사용되는 데이터셋의 경우, 미리 계산된 평균과 표준편차를 사용

In [ ]:
import torch
# torchvision: PyTorch에서 제공하는 컴퓨터 비전 라이브러리
    # 이미지 데이터셋, 모델 아키텍처, 이미지 변환 도구 등을 포함
import torchvision
import torchvision.transforms as transforms

# 데이터에 적용할 변환(transform) 정의
    # Compose: 여러 변환을 순차적으로 적용
transform = transforms.Compose([
    # 이미지를 텐서로 변환 (색상 정보 0~255를 0~1 사이 값으로 스케일링)
    transforms.ToTensor(), 
    # CIFAR-10 데이터셋의 R, G, B 채널별 평균과 표준편차로 정규화
    # mean=(R, G, B), std=(R, G, B)
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616))
])

# 훈련용(train) 데이터셋 로드
trainset = torchvision.datasets.CIFAR10(root=base_path + './data', train=True,
                                        download=True, transform=transform)
# 테스트용(test) 데이터셋 로드
testset = torchvision.datasets.CIFAR10(root=base_path + './data', train=False,
                                       download=True, transform=transform)

### 2-1-2. DataLoader 생성

- 전체 데이터셋을 미니배치 단위로 묶어주는 역할
- 전체 데이터를 한 번에 모델에 넣는 것은 메모리 부담이 크므로, 데이터를 작은 묶음으로 나눠서 학습 진행

In [ ]:
# 훈련용 DataLoader 생성
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                          shuffle=True)
# 테스트용 DataLoader 생성
testloader = torch.utils.data.DataLoader(testset, batch_size=128,
                                         shuffle=False)

## 2-2. CNN 모델 구현

- `nn.Module`을 상속받아서 CNN 모델 구조 설계
1. **특징 추출기**
    - 합성곱층과 풀링을 반복해서 이미지의 특징을 추출
    - 출력 크기 공식: $(W - F + 2P) / S + 1$
        - CIFAR10 이미지 크기: 32x32
        - W: 입력 크기 (높이 또는 너비)
        - F: 필터 크기
        - P: 패딩 크기
        - S: 스트라이드 (보폭)
2. **분류기**
    - 추출된 특징을 `Linear`계층을 통해 최종 클래스로 분류
    - 2-1에서 진행하였던 MLP

In [ ]:
import torch.nn as nn
import torch.nn.functional as F # ReLU 같은 활성화 함수들이 포함됨

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # --- 특징 추출기 (Feature Extractor) ---
        self.features = nn.Sequential(
            # 1. 첫 번째 합성곱 블록
                # 입력: 3채널(RGB), 출력: 16채널(필터 16개), 필터 크기: 3x3, 패딩: 1
                # 출력 크기: (32 - 3 + 2*1) / 1 + 1 = 32
                # 여기서 출력 크기란 특징 맵의 높이와 너비를 의미
                    # 즉 32x32 크기의 16채널 특징 맵이 생성됨
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            # 2x2 최대 풀링
                # 출력 크기: (32 - 2) / 2 + 1 = 16
            nn.MaxPool2d(kernel_size=2, stride=2),

            # 2. 두 번째 합성곱 블록
                # 입력: 16채널, 출력: 32채널, 필터 크기: 3x3, 패딩: 1
                # 출력 크기: (16 - 3 + 2*1) / 1 + 1 = 16
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # --- 분류기 (Classifier) ---
        # in_features 계산:
            # 필터 채널 수: 32

            # CIFAR10 이미지 크기: 32x32
                # conv1 통과 후: 32x32 (패딩=1이라 크기 유지)
                # pool 통과 후: 16x16 (크기 절반으로 감소)
                # conv2 통과 후: 16x16 (패딩=1이라 크기 유지)
                # pool 통과 후: 8x8 (크기 절반으로 감소)
            # 따라서, 32채널 x 8x8 = 2048
        self.classifier = nn.Sequential(
            nn.Linear(in_features=32 * 8 * 8, out_features=256),
            nn.ReLU(),
            nn.Linear(in_features=256, out_features=256),
            nn.ReLU(),
            # 최종 출력 레이어
                # out_features = 클래스 수 (num_classes)
            nn.Linear(in_features=256, out_features=num_classes)
        )

    def forward(self, x):
        # 1. 특징 추출
        x = self.features(x) 

        # 2. 평탄화 
            # nn.Flatten() 계층을 Sequential에 추가할 수도 있지만,
            # 배치 크기(-1) 처리를 위해 보통 forward에서 x.view()나 torch.flatten()을 사용합니다.
                # -1: 배치 크기 (자동 계산)
                # start_dim=1: 첫 번째 차원(채널)부터 평탄화
        x = torch.flatten(x, start_dim=1)  # (Batch_size, 32*8*8)로 변환

        # 3. 분류기 통과
        x = self.classifier(x)
        return x

## 2-3. 모델 학습 및 평가

### 2-3-1. 학습 준비

1. **장치 설정:** cuda 사용 설정
2. **모델 생성**
3. **손실 함수:** 다중 클래스 분류 문제이므로, `교차 엔트로피 오차` 사용
4. **옵티마이저:** 모델 가중치 업데이트 위한 Adam 사용 lr은 0.001로 설정

In [ ]:
# 장치 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

# 모델 생성 및 장치로 이동
model = SimpleCNN(num_classes=10).to(device)
# model = SimpleCNN(num_classes=10)

# 손실 함수와 옵티마이저 정의
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 2-3-2. 학습 및 평가 루프 구현

1. 10번 에포크 동안 학습
    - 1 에포크(전체 데이터 학습) 동안, 정해진 배치사이즈 만큼 나눠서 학습
    - 1 에포크 마다, 테스트 데이터로 성능 평가
2. `tqdm` 으로 진행률 시각화

In [ ]:
# 평가 진행률 시각화
from tqdm import tqdm

num_epochs = 10 # 10번 에포크 동안 학습

for epoch in range(num_epochs):
    # --- 훈련(Train) 단계 ---
    model.train() # 모델을 학습 모드로 설정
    running_loss = 0.0
    # 진행률 바 설정
    for i, data in enumerate(tqdm(trainloader), 0):
        inputs, labels = data[0].to(device), data[1].to(device)
        # inputs, labels = data[0], data[1]

        # 1. 옵티마이저의 기울기를 0으로 초기화
        optimizer.zero_grad()

        # 2. 순전파(forward pass)
        outputs = model(inputs)
        # 3. 손실 계산
        loss = criterion(outputs, labels)
        # 4. 역전파(backward pass)
        loss.backward()
        # 5. 파라미터 업데이트
        optimizer.step()

        running_loss += loss.item()

    print(f'[{epoch + 1}] 훈련 손실: {running_loss / len(trainloader):.3f}')

    # --- 평가(Evaluation) 단계 ---
    model.eval() # 모델을 평가 모드로 설정
    correct = 0
    total = 0
    with torch.no_grad(): # 기울기 계산 비활성화
        for data in testloader:
            images, labels = data[0].to(device), data[1].to(device)
            # images, labels = data[0], data[1]
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'[{epoch + 1}] 테스트 정확도: {100 * correct / total:.2f} %')

print('학습 종료')

# 99. Extra - 실제 이미지를 분류 할 수 있을까?

In [ ]:
from PIL import Image

# convert('RGB')는 이미지가 4채널(RGBA)일 경우를 대비해 3채널(RGB)로 통일해주는 역할
image = Image.open(base_path + 'cat.jpg').convert('RGB')

# 이미지 확인
# image.show() 

# 훈련 시 사용했던 전처리 과정을 그대로 정의
inference_transform = transforms.Compose([
    transforms.Resize((32, 32)), # 1. 32x32 크기로 리사이즈
    transforms.ToTensor(),       # 2. 텐서로 변환
    # 3. 훈련 때와 "동일한" 값으로 정규화
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616))
])

# 전처리 적용
input_tensor = inference_transform(image)

# 전처리 된 이미지 확인
print(input_tensor.shape) # torch.Size([3, 32, 32])

## 99-2. 예측 및 결과 확인

1. `unsqueeze(0)`
    - 모델은 기본적으로 데이터가 **배치 단위**로 들어올 것이라 학습되었음.
    - 그러나 실제 이미지는 1장이므로, `[채널, 높이, 너비]` 형태의 텐서에 배치 차원을 추가
    - `[1, 채널, 높이, 너비]` 형태로 변환하여 진행
2. `model.eval()`
    - 모델의 평가 모드
    - 드랍아웃과 같은 훈련용 기능들을 비활성화 함

In [ ]:
import torch

# 1. 모델을 평가 모드로 전환
model.eval()

# 2. 배치 차원 추가 및 장치로 이동
input_batch = input_tensor.unsqueeze(0).to(device)
# input_batch = input_tensor.unsqueeze(0)

# 3. 기울기 계산을 하지 않도록 설정
with torch.no_grad():
    output = model(input_batch)

# 4. 출력(Logits)을 확률(Probabilities)로 변환
# Softmax 함수는 모든 클래스에 대한 점수의 합이 1이 되도록 만듬
probabilities = torch.nn.functional.softmax(output[0], dim=0)

# 5. 가장 확률이 높은 클래스 찾기
top_prob, top_catid = torch.max(probabilities, 0)
predicted_idx = top_catid.item()

# CIFAR-10 클래스 이름 가져오기
class_names = trainset.classes # ['airplane', 'automobile', 'bird', 'cat', ...]
predicted_label = class_names[predicted_idx]

print(f"모델의 예측: '{predicted_label}' (신뢰도: {top_prob.item()*100:.2f}%)")

# 시각화로 최종 확인
import matplotlib.pyplot as plt

plt.imshow(image)
plt.title(f"Prediction: {predicted_label} ({top_prob.item()*100:.2f}%)")
plt.axis('off')
plt.show()
